# 不正検知② — 改善モデル：2系統化と未知手口への対応をテストで測る

出典：第02章 03〜04節／パス：`02_fraud/02_improved.ipynb`／パッケージ：numpy・pandas・scikit-learn／実行時間の目安：約2分

このノート単体で実行できる。`input/transactions_modus.csv`（手口ラベル付きの合成データ）を読み込む。欠品時は内蔵生成へ切り替わる。
ベースラインのコードは `01_baseline.ipynb` に分けてある。旗上げ率2%は審査キャパシティの仮置き（テストを見て決めた値ではない）。最終評価はテストセットで1回だけ行う。


テストは「既知データ（正常＋手口A）のうち学習に使わなかった25%」と「手口Bの全件」に限る。学習に使った行はテストに入れない。

In [1]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RS = 0


def make_fraud_data2(n: int, seed: int = 7) -> pd.DataFrame:
    r = np.random.default_rng(seed)
    amount = np.clip(r.lognormal(mean=3.0, sigma=1.0, size=n), 1.0, 500.0)
    hour = r.integers(0, 24, size=n)
    night = ((hour <= 4) | (hour >= 23)).astype(float)
    intl = (r.uniform(size=n) < 0.08).astype(int)
    card_age = np.clip(r.exponential(scale=24.0, size=n), 0.0, 120.0)
    new_card = (card_age < 3.0).astype(float)
    online = (r.uniform(size=n) < 0.60).astype(int)
    merchant = r.choice(["retail", "food", "travel", "digital", "other"], size=n,
                       p=[0.35, 0.25, 0.10, 0.15, 0.15])
    risky_merch = np.isin(merchant, ["travel", "digital"]).astype(float)
    # 手口A：海外×新規カード ／ 手口B：深夜×高額デジタル（新手口として後から現れる）
    logit_a = -5.4 + 3.0 * intl * new_card + 0.008 * amount * intl - 0.02 * card_age
    logit_b = -5.4 + 0.030 * amount * night * risky_merch * online + 1.6 * new_card * night
    pa = 1.0 / (1.0 + np.exp(-logit_a))
    pb = 1.0 / (1.0 + np.exp(-logit_b))
    is_a = (r.uniform(size=n) < pa).astype(int)
    is_b = ((1 - is_a) * (r.uniform(size=n) < pb)).astype(int)
    fraud = np.maximum(is_a, is_b)
    modus = np.where(is_a == 1, "A", np.where(is_b == 1, "B", "-"))
    return pd.DataFrame({
        "amount": amount, "hour": hour, "intl": intl, "card_age": card_age,
        "online": online, "merchant": merchant, "fraud": fraud, "modus": modus,
    })


INPUT = "input/transactions_modus.csv"
try:
    df = pd.read_csv(INPUT)
    print(f"Inputを使用: {INPUT}")
except FileNotFoundError:
    df = make_fraud_data2(20000)
    print("Inputが無いため合成データを生成")
print(f"件数: {len(df)}, 不正率: {df['fraud'].mean():.4f}")
print(df[df['fraud'] == 1]['modus'].value_counts().to_string())

Inputを使用: input/transactions_modus.csv
件数: 20000, 不正率: 0.0095
modus
B    122
A     67


In [2]:
# 設定：学習時は手口Aしか見ていない（手口Bは未知の新手口としてテストにだけ出す）
FEATS = ["amount", "hour", "intl", "card_age", "online", "merchant"]
known = df[df["modus"].isin(["-", "A"])]
k_tr, k_te = train_test_split(known, test_size=0.25, stratify=known["fraud"], random_state=RS)
# テスト＝既知データの残り25%（学習に使っていない）＋手口Bの全件
test = pd.concat([k_te, df[df["modus"] == "B"]])
X_tr, y_tr = k_tr[FEATS], k_tr["fraud"]
X_te, y_te = test[FEATS], test["fraud"]
modus_te = test["modus"].to_numpy()
is_known = modus_te != "B"
print(f"学習: {len(X_tr)}件（手口A {int(y_tr.sum())}件）/ テスト: {len(X_te)}件"
      f"（手口A {int((modus_te == 'A').sum())}件・手口B {int((modus_te == 'B').sum())}件）")
num_cols = ["amount", "hour", "intl", "card_age", "online"]
pre = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["merchant"]),
])
sup = make_pipeline(
    clone(pre),
    HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05,
                                   max_leaf_nodes=15, min_samples_leaf=50, random_state=RS),
).fit(X_tr, y_tr)
p_sup = sup.predict_proba(X_te)[:, 1]
# 比較の基準：同じ学習データでの線形（ベースラインと同じ手法）
lr = make_pipeline(clone(pre), LogisticRegression(max_iter=5000)).fit(X_tr, y_tr)
p_lr = lr.predict_proba(X_te)[:, 1]
# 異常検知：正常だけを学ぶ（IsolationForest。汚染率は業務知識の仮置き0.02）
pre_iso = clone(pre).fit(X_tr)
iso = IsolationForest(n_estimators=200, contamination=0.02, random_state=RS)
iso.fit(pre_iso.transform(X_tr)[y_tr.to_numpy() == 0])
s_iso = -iso.decision_function(pre_iso.transform(X_te))  # 大きいほど異常
# PR-AUCは2通り：既知（正常＋手口A）だけで測る値と、手口A＋Bを合わせた全不正で測る値
print("PR-AUC（既知＝正常＋手口A / 全体＝手口A＋B）:")
for name, s in [("ベースライン・線形", p_lr), ("教師あり", p_sup), ("異常検知", s_iso)]:
    print(f"[{name}] 既知={average_precision_score(y_te[is_known], s[is_known]):.4f}"
          f" / 全体={average_precision_score(y_te, s):.4f}")

学習: 14908件（手口A 50件）/ テスト: 5092件（手口A 17件・手口B 122件）


PR-AUC（既知＝正常＋手口A / 全体＝手口A＋B）:
[ベースライン・線形] 既知=0.0080 / 全体=0.0309
[教師あり] 既知=0.0421 / 全体=0.0377
[異常検知] 既知=0.0072 / 全体=0.0509


In [3]:
# 未知の手口Bに対する再現率（旗上げ率2%は運用キャパシティの仮置きで固定して比較）
for name, s in [("ベースライン・線形", p_lr), ("教師あり", p_sup), ("異常検知", s_iso)]:
    thr = np.quantile(s, 0.98)
    flag = s >= thr
    out = []
    for m in ["A", "B"]:
        mm = modus_te == m
        out.append(f"手口{m}の再現率={flag[mm].mean():.3f}")
    print(f"[{name}] " + " / ".join(out) + f" / 全体の旗上げ率={flag.mean():.3f}")


[ベースライン・線形] 手口Aの再現率=0.059 / 手口Bの再現率=0.016 / 全体の旗上げ率=0.020
[教師あり] 手口Aの再現率=0.235 / 手口Bの再現率=0.008 / 全体の旗上げ率=0.021
[異常検知] 手口Aの再現率=0.059 / 手口Bの再現率=0.074 / 全体の旗上げ率=0.020


In [4]:
# 閾値運用：スコアを残し、旗上げ率と再現率の対応表で運用点を選ぶ
yv = y_te.to_numpy()
print("旗上げ率 / 適合率 / 再現率:")
for q in [0.995, 0.99, 0.98, 0.95, 0.90]:
    thr = np.quantile(s_iso, q)
    flag = s_iso >= thr
    tp = int((flag & (yv == 1)).sum())
    fp = int((flag & (yv == 0)).sum())
    fn = int(((~flag) & (yv == 1)).sum())
    print(f"{(1 - q):.3f} / {tp / (tp + fp):.3f} / {tp / (tp + fn):.3f}")
assert np.isfinite(s_iso).all()

旗上げ率 / 適合率 / 再現率:
0.005 / 0.000 / 0.000
0.010 / 0.098 / 0.036
0.020 / 0.098 / 0.072
0.050 / 0.075 / 0.137
0.100 / 0.067 / 0.245


In [5]:
# 改善結果：ベースライン（同じデータの線形）と改善モデルの差をテストで1回だけ測る
ap = lambda s, m: average_precision_score(y_te[m], s[m])
print(f"[改善結果] 既知手口（正常＋手口A）のPR-AUC {ap(p_lr, is_known):.4f} → {ap(p_sup, is_known):.4f}"
      f"（{ap(p_sup, is_known) - ap(p_lr, is_known):+.4f}）")
for name, s in [("ベースライン・線形", p_lr), ("異常検知", s_iso)]:
    thr = np.quantile(s, 0.98)
    print(f"[改善結果] 未知手口Bの再現率（旗上げ2%）: {name}={(s >= thr)[modus_te == 'B'].mean():.3f}")

[改善結果] 既知手口（正常＋手口A）のPR-AUC 0.0080 → 0.0421（+0.0341）
[改善結果] 未知手口Bの再現率（旗上げ2%）: ベースライン・線形=0.016
[改善結果] 未知手口Bの再現率（旗上げ2%）: 異常検知=0.074


In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
_book_fig = Path("../../../books/python-machine-jitumu/assets/images")
FIG = str(_book_fig if _book_fig.is_dir() else Path("outputs"))
Path(FIG).mkdir(parents=True, exist_ok=True)
names, a_rec, b_rec = [], [], []
for name, s in [("linear", p_lr), ("supervised", p_sup), ("anomaly", s_iso)]:
    flag = s >= np.quantile(s, 0.98)
    names.append(name)
    a_rec.append(flag[modus_te == "A"].mean())
    b_rec.append(flag[modus_te == "B"].mean())
x = np.arange(len(names)); w = 0.35
plt.figure(); plt.bar(x - w / 2, a_rec, w, label="modus A (known)")
plt.bar(x + w / 2, b_rec, w, label="modus B (unknown)")
plt.xticks(x, names); plt.ylabel("recall"); plt.legend(); plt.title("recall by modus (test)")
plt.savefig(f"{FIG}/code-result-j02-02.png", dpi=150, bbox_inches="tight"); plt.close()
print("saved code-result-j02-02.png")


saved code-result-j02-02.png


In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
_book_fig = Path("../../../books/python-machine-jitumu/assets/images")
FIG = str(_book_fig if _book_fig.is_dir() else Path("outputs"))
Path(FIG).mkdir(parents=True, exist_ok=True)
from sklearn.metrics import precision_recall_curve
plt.figure()
for name, s in [("baseline", p_lr), ("supervised", p_sup), ("anomaly", s_iso)]:
    prec, rec, _ = precision_recall_curve(y_te, s)
    plt.step(rec, prec, where="post", label=name)
plt.xlabel("recall"); plt.ylabel("precision"); plt.legend()
plt.title("PR curves before vs after (test: holdout + modus B)")
plt.savefig(f"{FIG}/code-result-j02-03.png", dpi=150, bbox_inches="tight"); plt.close()
print("saved code-result-j02-03.png")


saved code-result-j02-03.png


## 読み取りどころ

- 教師ありは既知の手口Aに強いが、未知の手口Bで落ちること。
- 異常検知は全体精度では劣るが、手口Bにも一定反応すること（正常の外れを見る強み）。
- 汚染率・分位点は閾値そのもの——スコアを残して運用で選ぶこと。